## Responses API

- https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses?tabs=python-key

In [3]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"] + "/openai/v1/",
)

response = client.responses.create(   
    model="gpt-5",
    input="How many r in strawberrier?",
    reasoning={
        "effort": "low",  # minimal/low/medium/high
        "summary": "concise"  # auto/concise/detailed
    }
)

In [6]:
def print_response(response, raw=False):
    if raw:
        print(response.model_dump_json(indent=2))
        return

    print("response id:", response.id)
    for output in response.output:
        if output.type == "message":
            print("message:", "\n".join([c.text for c in output.content]))
        elif output.type == "reasoning":
            if output.summary:
                print("reasoning:", "\n".join([s.text for s in output.summary]))
        elif output.type == "function_call":
            print("function call:", output.name, output.arguments, output.call_id)
        elif output.type == "web_search_call":
            print("web search:", output.action)
        elif output.type == "code_interpreter_call":
            print("code:\n", output.code)
        else:
            print("output:", output.model_dump())

In [15]:
print_response(response)

response id: resp_0304a4c0a96f9c8b0069b7ab288cb081959f2c557f5c6cc1be
reasoning: **Counting letters in 'strawberrier'**

I need to answer the question about how many "r"s are in "strawberrier." It seems a bit tricky! First, I can break down the spelling: s, t, r, a, w, b, e, r, r, i, e, r. Counting the "r"s, I find there are four of them: r (1), r (2), r (3), and r (4). Even though they might mean "strawberries" or "strawberry," since the question specifies "strawberrier," I’ll just confirm there are four "r"s in that word.
message: 4


In [18]:
# chain responses together using previous_response_id
response2 = client.responses.create(   
    model="gpt-5",
    input="How about roar?",
    previous_response_id=response.id,
)

print_response(response2)

response id: resp_0304a4c0a96f9c8b0069b7aedf6894819592c990f7020f3060
reasoning: 
message: 2


In [6]:
# retrieve chained input items for a response
input_items = client.responses.input_items.list(response2.id)

for item in input_items.data:
    print(item.model_dump(include=["type", "content", "summary"]))

{'content': [{'text': 'How about roar?', 'type': 'input_text'}], 'type': 'message'}
{'content': None, 'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': '**Counting letters in "strawberrier"**\n\nI need to figure out what the user means by asking about the number of \'r\'s in "strawberrier." It seems like it might be a misspelling of "strawberries," but since they provided that specific word, I\'ll focus on it. \n\nBreaking it down, I see there are four occurrences of the letter \'r\' in "strawberrier," found at positions 3, 8, 9, and 12. If they meant "strawberries," there would only be three \'r\'s.'}, {'type': 'summary_text', 'text': '**Clarifying the letter count**\n\nOkay, let’s look at both "strawberries" and "strawberry." \n\nFor "strawberries," I see \'r\' in positions 3, 8, and 9, totaling three \'r\'s. \n\nFor "strawberry," checking again, I confirm there are also three \'r\'s in positions 3, 8, and 9. \n\nHowever, since the user asked about "strawberrier," the

### Function Calling

In [10]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the weather for a location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
            },
            "required": ["location"],
        },
    }
]
response = client.responses.create(  
    model="gpt-5",
    tools=tools,
    input="What's the weather in Dehradun?"
)

print_response(response)

response id: resp_0df89f1e68bd0f2a0069b6cd62ac708195adbc7f0e1ebb6c6a
reasoning: 
function call: get_weather {"location":"Dehradun"} call_TOfg9J16aB7jEfUYl0Bu0mCs


In [12]:
response2 = client.responses.create(
    model="gpt-5",
    previous_response_id=response.id,
    tools=tools,
    input=[
        { "type": "function_call_output", "call_id": response.output[1].call_id, "output": '{"temperature": "20 C"}' },
        { "role": "user", "content": "How about Gairsain?" }
    ],
)

print_response(response2)

response id: resp_0df89f1e68bd0f2a0069b6cd7b11208195b37676cb5a2c1a95
reasoning: 
function call: get_weather {"location":"Gairsain"} call_8YRq08fuIfcCWVyHRKU1aFbg


### Web search

In [22]:
response = client.responses.create(  
    model="gpt-5",
    tools=[{ "type": "web_search" }],
    input="Who won the 2026 T20 World Cup?"
)

print_response(response)

response id: resp_0a63501cc0fbed900069b7d4f1613481939a00149e50db0a9a
message: India. They defeated New Zealand by 96 runs in the final on March 8, 2026, at Narendra Modi Stadium in Ahmedabad. ([icc-cricket.com](https://www.icc-cricket.com/tournaments/mens-t20-world-cup-2026/news/live-india-new-zealand-chase-world-cup-glory))


### Code interpreter

In [46]:
response = client.responses.create(  
    model="gpt-5",
    tools=[
        { "type": "code_interpreter", "container": { "type": "auto" } },
        { "type": "web_search" },
    ],
    input="Fetch the latest stock prices of NVIDIA for the last 7 days and compute DoD change. Save it to a csv file.",
    reasoning={
        "effort": "low",
    }
)

print_response(response)

response id: resp_0b18443d612cbb110069b7deccac0c81979908e4041d688bbb
web search: ActionSearch(query='NVDA historical data Yahoo Finance', type='search', queries=['NVDA historical data Yahoo Finance', 'NVIDIA historical prices last 7 days', 'NVDA historical data Nasdaq', 'NVDA historical data Google Finance'], sources=None)
web search: ActionOpenPage(type='open_page', url='https://finance.yahoo.com/quote/NVDA/history/')
web search: ActionSearch(query='NVDA historical data Nasdaq', type='search', queries=['NVDA historical data Nasdaq', 'NVDA historical data site:nasdaq.com'], sources=None)
web search: ActionOpenPage(type='open_page', url='https://www.macrotrends.net/stocks/charts/NVDA/nvidia/stock-price-history')
web search: ActionOpenPage(type='open_page', url='https://stockanalysis.com/stocks/nvda/history/')
code:
 import pandas as pd

# Manually entered from StockAnalysis.com (NVDA daily closes)
data = [
    {"Date": "2026-03-12", "Close": 183.13},
    {"Date": "2026-03-11", "Close": 

In [47]:
assistant_output = [o for o in response.output if o.type == "message" and o.role == "assistant"][0]
container_file_citation = [a for a in assistant_output.content[0].annotations if a.type == "container_file_citation"][0]
container_id = container_file_citation.container_id
file_id = container_file_citation.file_id

container_id, file_id

('cntr_69b7decce01881908f566819fefd97e1097c53364649ef40',
 'cfile_69b7defaacc08190bc10ea3fb169d388')

In [48]:
content = client.containers.files.content.retrieve(
    container_id=container_id,
    file_id=file_id,
)

print(content.content.decode("utf-8"))

Date,Close,DoD_Change,DoD_Change_Pct
2026-03-12,183.13,-2.9000000000000057,-1.558888351341181
2026-03-11,186.03,1.259999999999991,0.6819288845591798
2026-03-10,184.77,2.1200000000000045,1.160689843963869
2026-03-09,182.65,4.8300000000000125,2.7162298953998576
2026-03-06,177.82,-5.52000000000001,-3.010799607287018
2026-03-05,183.34,0.30000000000001137,0.16389860139860435
2026-03-04,183.04,,



### MCP

To setup the MCP server checkout `mcp_server/` folder, and update `.env` with MCP endpoint and API key. 

In [11]:
mcp_server_url = os.environ["MCP_ENDPOINT"].rstrip("/") + "/mcp"
mcp_api_key = os.environ["MCP_API_KEY"]
response = client.responses.create(
    model="gpt-5",
    tools=[
        {
            "type": "mcp",
            "server_label": "retail-mcp",
            "server_url": mcp_server_url,
            "headers": {
                "X-MCP-API-Key": mcp_api_key
            },
            "require_approval": "never",
        }
    ],
    input=[
        {
            "role": "system",
            "content": "You are a helpful Retail Support Agent that helps customers with their orders, exchanges, returns and refunds.",
        },
        {
            "role": "user",
            "content": "Compute and share the total paid amount for all my orders. My account email is ava.moore2222@example.com",
        },
    ],
)

print_response(response)

response id: resp_0df0cb8164e324f10069e35ca3876c8196bd8dc7bf62c5c203
output: {'id': 'mcpl_0df0cb8164e324f10069e35ca394688196a83486500bfb34fe', 'server_label': 'retail-mcp', 'tools': [{'input_schema': {'properties': {'expression': {'title': 'Expression', 'type': 'string'}}, 'required': ['expression'], 'title': 'calculateArguments', 'type': 'object'}, 'name': 'calculate', 'annotations': {'read_only': False}, 'description': 'Calculate the result of a mathematical expression.'}, {'input_schema': {'properties': {'email': {'title': 'Email', 'type': 'string'}}, 'required': ['email'], 'title': 'find_user_id_by_emailArguments', 'type': 'object'}, 'name': 'find_user_id_by_email', 'annotations': {'read_only': False}, 'description': 'Find user id by email.'}, {'input_schema': {'properties': {'first_name': {'title': 'First Name', 'type': 'string'}, 'last_name': {'title': 'Last Name', 'type': 'string'}, 'zip': {'title': 'Zip', 'type': 'string'}}, 'required': ['first_name', 'last_name', 'zip'], 'titl